# Lab 2：PyTorch 基础与螺旋数据分类

本 Notebook 对应课程实验二，包含 Tensor 基础、螺旋数据可视化、线性模型与带 ReLU 的 MLP 对比。代码不依赖本地文件，上传 Colab 后可以直接运行。

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(7)
np.random.seed(7)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

a = torch.arange(6).reshape(2, 3)
print('tensor:', a)
print('shape:', a.shape, 'dtype:', a.dtype)

In [ ]:
def make_spiral(points_per_class=300, classes=3):
    X = np.zeros((points_per_class * classes, 2), dtype=np.float32)
    y = np.zeros(points_per_class * classes, dtype=np.int64)
    for cls in range(classes):
        idx = slice(cls * points_per_class, (cls + 1) * points_per_class)
        radius = np.linspace(0.05, 1.0, points_per_class)
        theta = np.linspace(cls * 4.0, (cls + 1) * 4.0, points_per_class)
        theta += np.random.randn(points_per_class) * 0.18
        X[idx] = np.c_[radius * np.sin(theta), radius * np.cos(theta)]
        y[idx] = cls
    return torch.tensor(X), torch.tensor(y)

X, y = make_spiral()
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=8)
plt.title('三分类螺旋数据')
plt.show()

In [ ]:
class SpiralMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        return self.net(x)

model = SpiralMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
criterion = nn.CrossEntropyLoss()
X_device, y_device = X.to(device), y.to(device)
loss_history = []
for epoch in range(500):
    optimizer.zero_grad()
    logits = model(X_device)
    loss = criterion(logits, y_device)
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if (epoch + 1) % 100 == 0:
        pred = logits.argmax(dim=1)
        acc = (pred == y_device).float().mean().item()
        print('epoch', epoch + 1, 'loss', round(loss.item(), 4), 'acc', round(acc, 4))

plt.plot(loss_history)
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('Adam 优化螺旋分类损失')
plt.show()

## 小结

线性层只能学习线性边界，螺旋数据需要 ReLU 引入非线性；优化器也会影响收敛速度。报告中记录的 SGD、Adam 和 AdamW 对比，可以在这段代码中替换 `optimizer` 后复现。